# Medicare IP Model Training — Full Downstream Evaluation

## Three-Model Comparison: New TE vs New TE + Baseline vs Production

**Purpose**: Compare three data source configurations for Medicare IP (6-month) prediction:

| Model | Embedding Source | Baseline Features | Description |
|-------|-----------------|-------------------|-------------|
| **Model 1** | New TE (256 dims) | None | Exp Round 10 / Exp2b embeddings only |
| **Model 2** | New TE (48 selected dims) | 52 production features | New embeddings + production baseline |
| **Model 3** | Production TE (48 dims) | 52 production features | Current production model configuration |

**Temporal Design**:
- Training period: 2024-07-01 to 2025-06-30 (random stratified 70/15/15 train/val/test)
- Out-of-Time (OOT) test: 2025-07-01 to 2025-09-30

**Outcome**: `ip6` — binary inpatient admission within 6 months

**Model**: XGBoost binary classifier

In [ ]:
import os
import pandas as pd
import numpy as np
import pickle
import seaborn as sns
import matplotlib.pyplot as plt
import random
import warnings
import joblib
import gc
from io import BytesIO

from sklearn import preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, roc_auc_score, roc_curve, confusion_matrix,
    classification_report, brier_score_loss, precision_recall_curve,
    auc, precision_score, average_precision_score
)

import xgboost as xgb
from xgboost import XGBClassifier

from scipy.stats import percentileofscore

from google.cloud import storage
from google.cloud import bigquery
import google.auth

import time
from datetime import datetime

random.seed(123)
np.random.seed(123)
warnings.filterwarnings("ignore")
client = bigquery.Client()

## Configuration

In [ ]:
# =============================================================================
# TABLE CONFIGURATION
# =============================================================================

# New TE embeddings (Exp Round 10 / Exp2b: flash attention + learned pooling v2)
NEW_TE_TABLE = 'edp-prod-storage.edp_ent_sdoheir_cns.a834793_exp_round10_exp2b_medicare_embeddings_20241120_20250930'

# Production features + production TE embeddings
PRODUCTION_TABLE = 'anbc-hcb-prod.clin_analytics_hcb_prod.inpatient_me_features_history'

# Outcomes table (generated by medicare_ip_outcome_generation_exp_round10_exp2b.sql)
OUTCOMES_TABLE = 'edp-prod-storage.edp_ent_sdoheir_cns.a834793_Medicare_outcome_6mo_final_exp_round10_exp2b'

# =============================================================================
# TEMPORAL CONFIGURATION
# =============================================================================

# Training period: train/val/test with random stratified split
TRAIN_START = '2024-07-01'
TRAIN_END   = '2025-06-30'

# Out-of-time test period
OOT_START = '2025-07-01'
OOT_END   = '2025-09-30'

# =============================================================================
# FEATURE CONFIGURATION
# =============================================================================

# 48 production-selected embedding dimensions (same indices for new TE and production TE)
PRODUCTION_EMBEDDINGS = [
    'embedding_6', 'embedding_7', 'embedding_14', 'embedding_20', 'embedding_23',
    'embedding_30', 'embedding_31', 'embedding_36', 'embedding_43', 'embedding_47',
    'embedding_49', 'embedding_57', 'embedding_64', 'embedding_76', 'embedding_81',
    'embedding_89', 'embedding_94', 'embedding_95', 'embedding_98', 'embedding_104',
    'embedding_110', 'embedding_111', 'embedding_124', 'embedding_126', 'embedding_127',
    'embedding_130', 'embedding_131', 'embedding_138', 'embedding_173', 'embedding_174',
    'embedding_177', 'embedding_178', 'embedding_188', 'embedding_192', 'embedding_195',
    'embedding_203', 'embedding_205', 'embedding_207', 'embedding_212', 'embedding_219',
    'embedding_224', 'embedding_229', 'embedding_230', 'embedding_233', 'embedding_238',
    'embedding_244', 'embedding_253', 'embedding_254'
]

# Mapping: production table emb columns -> standardized embedding column names
# Production table uses 'emb6', 'emb7', etc. -> we rename to 'embedding_6', 'embedding_7', etc.
PROD_EMB_COL_MAP = {f'emb{col.split("_")[1]}': col for col in PRODUCTION_EMBEDDINGS}
# e.g., {'emb6': 'embedding_6', 'emb7': 'embedding_7', ...}

# All 256 new TE embedding columns
ALL_NEW_TE_EMBEDDINGS = [f'embedding_{i}' for i in range(256)]

# 52 baseline features (same as production model)
BASELINE_FEATURES = [
    # HPD Chronic Conditions (11)
    'camemhpd_aff', 'camemhpd_alc', 'camemhpd_cbd', 'camemhpd_chf', 'camemhpd_chr_flag',
    'camemhpd_cop', 'camemhpd_crf', 'camemhpd_cv_cond', 'camemhpd_dia', 'camemhpd_hyp', 'camemhpd_ngd',
    # Medical Utilization (4)
    'camemmedutilization_clm_ln_cnt', 'camemmedutilization_er_clm_cnt',
    'camemmedutilization_uniq_dx_cd_cnt', 'camemmedutilization_uniq_rev_cd_cnt',
    # Medical Case (5)
    'camemmedcasedc1_ip_cnt_dc1', 'camemmedcasedc1_ip_days_dc1',
    'camemmedcasedc2_ip_cnt_dc2', 'camemmedcasedc2_ip_days_dc2', 'camemmedcasedc3_ip_cnt_dc3',
    # Diagnosis (1)
    'camemeipdxdc1_dxc1085_cnt_dc1',
    # Procedures (5)
    'camemeipprcdc1_prc141_cnt_dc1', 'camemeipprcdc1_prc155_cnt_dc1',
    'camemeipprcdc1_prc219_cnt_dc1', 'camemeipprcdc1_prcc1102_cnt_dc1', 'camemeipprcdc1_prcc1115_cnt_dc1',
    # Revenue Codes (2)
    'camemrevenuedc3_rev730_cnt_dc3', 'camemrevenuedc4_rev430_cnt_dc4',
    # ER/UC (2)
    'camemerucdc1_erclm_cnt_dc1', 'camemerucdc2_erclm_cnt_dc2',
    # RX Class (2)
    'camemrxclassutilizationdc5_anticonvulsants_misc_flag_dc5',
    'camemrxclassutilizationdc5_loop_diuretics_flag_dc5',
    # RX Group (5)
    'camemrxgrouputilizationdc1_antidepressants_days_dc1',
    'camemrxgrouputilizationdc1_corticosteroids_days_dc1',
    'camemrxgrouputilizationdc2_anticonvulsants_days_dc2',
    'camemrxgrouputilizationdc2_corticosteroids_days_dc2',
    'camemrxgrouputilizationdc3_anticonvulsants_flag_dc3',
    # Specialty Claims (3)
    'camemspcclmdc1_spcclmwhos_cnt_dc1', 'camemspcclmdc2_spcclmwhos_cnt_dc2',
    'camemspcofcdc3_spcd_cnt_dc3',
    # Care Management (2)
    'camemtgtptpdc2_cm_soe', 'camemtgtptpdc5_cm_soe',
    # Text Notes (2)
    'camemtxtnotesdc5_txt_end_dc5', 'camemtxtnotesdc5_txt_short_dc5',
    # YLM Demographics (4)
    'camemylm_ylm_homeagesourcer', 'camemylm_ylm_orent',
    'camemylm_ylm_tw_hvalsecinv', 'camemylm_ylm_tw_hvyinvtrad',
    # Membership Demographics (2)
    'camemmbrshp_age65_74', 'camemmbrshp_agenbr',
    # Expenditure (2)
    'e_caperetdem21220', 'e_caperetdem444'
]

print(f'Production selected embeddings: {len(PRODUCTION_EMBEDDINGS)}')
print(f'All new TE embeddings: {len(ALL_NEW_TE_EMBEDDINGS)}')
print(f'Baseline features: {len(BASELINE_FEATURES)}')
print(f'Production embedding column map: {list(PROD_EMB_COL_MAP.items())[:5]}...')

## Helper Functions

In [ ]:
def preprocess_dataframe(df):
    """Handle missing values and type conversions (same as production pipeline)"""
    print(f"Preprocessing dataframe: {df.shape}")

    # Convert target to int
    if 'ip6' in df.columns:
        df['ip6'] = df['ip6'].astype('int')

    # Handle float32 columns
    float32_columns = df.select_dtypes(include=['float32']).columns
    df[float32_columns] = df[float32_columns].fillna(0.0)
    for col in float32_columns:
        df[col] = df[col].astype('float')

    # Handle float64 columns
    float64_columns = df.select_dtypes(include=['float64']).columns
    df[float64_columns] = df[float64_columns].fillna(0.0)
    for col in float64_columns:
        df[col] = df[col].astype('float')

    # Handle int64 columns
    int64_columns = df.select_dtypes(include=['int64']).columns
    df[int64_columns] = df[int64_columns].fillna(0)
    for col in int64_columns:
        df[col] = df[col].astype('int')

    # Handle object columns (empty string -> 0, NaN -> 0, cast to int)
    df = df.replace('', 0)
    object_columns = df.select_dtypes(include=['object']).columns
    df[object_columns] = df[object_columns].fillna(0)
    for col in object_columns:
        try:
            df[col] = df[col].astype('int')
        except:
            print(f"  Warning: Could not convert {col} to int, keeping as object")

    print(f"Preprocessing complete: {df.shape}")
    return df

In [ ]:
def lift_at_percentage(y_true, y_prob, pct):
    """Lift = precision@k / baseline_prevalence."""
    n = len(y_true)
    k = max(1, int(n * pct))
    top_k_indices = np.argsort(y_prob)[::-1][:k]
    precision_at_k = y_true[top_k_indices].mean()
    baseline = y_true.mean()
    return precision_at_k / baseline if baseline > 0 else 0.0


def true_positives_at_percentage(y_true, y_prob, pct):
    """Count of true positives in top-k% of predictions."""
    n = len(y_true)
    k = max(1, int(n * pct))
    top_k_indices = np.argsort(y_prob)[::-1][:k]
    return int(y_true[top_k_indices].sum())


def precision_at_percentage(y_true, y_prob, pct):
    """Precision in top-k% of predictions."""
    n = len(y_true)
    k = max(1, int(n * pct))
    top_k_indices = np.argsort(y_prob)[::-1][:k]
    return float(y_true[top_k_indices].mean())


def calculate_comprehensive_metrics(y_true, y_pred_proba, set_name='test'):
    """Calculate all evaluation metrics for a given set."""
    y_true = np.array(y_true)
    y_pred_proba = np.array(y_pred_proba)

    metrics = {
        'auc_roc':         roc_auc_score(y_true, y_pred_proba),
        'auc_pr':          average_precision_score(y_true, y_pred_proba),
        'brier':           brier_score_loss(y_true, y_pred_proba),
        'lift_1pct':       lift_at_percentage(y_true, y_pred_proba, 0.01),
        'lift_5pct':       lift_at_percentage(y_true, y_pred_proba, 0.05),
        'lift_10pct':      lift_at_percentage(y_true, y_pred_proba, 0.10),
        'tp_1pct':         true_positives_at_percentage(y_true, y_pred_proba, 0.01),
        'precision_1pct':  precision_at_percentage(y_true, y_pred_proba, 0.01),
        'tp_5pct':         true_positives_at_percentage(y_true, y_pred_proba, 0.05),
        'precision_5pct':  precision_at_percentage(y_true, y_pred_proba, 0.05),
        'tp_10pct':        true_positives_at_percentage(y_true, y_pred_proba, 0.10),
        'precision_10pct': precision_at_percentage(y_true, y_pred_proba, 0.10),
        'n_samples':       len(y_true),
        'n_positives':     int(y_true.sum()),
        'prevalence':      float(y_true.mean()),
    }

    prefixed_metrics = {f"{set_name}_{k}": v for k, v in metrics.items()}
    return prefixed_metrics


def print_metrics_summary(metrics, model_name):
    """Print formatted metrics summary for a model."""
    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")
    for set_name in ['train', 'val', 'test']:
        prefix = f"{set_name}_"
        set_metrics = {k.replace(prefix, ''): v for k, v in metrics.items() if k.startswith(prefix)}
        if not set_metrics:
            continue
        print(f"\n  {set_name.upper()} Set (n={set_metrics.get('n_samples', '?')}, pos={set_metrics.get('n_positives', '?')}, prev={set_metrics.get('prevalence', 0):.4f}):")
        print(f"    AUC-ROC: {set_metrics.get('auc_roc', 0):.4f}")
        print(f"    AUC-PR:  {set_metrics.get('auc_pr', 0):.4f}")
        print(f"    Brier:   {set_metrics.get('brier', 0):.4f}")
        print(f"    Lift@1%: {set_metrics.get('lift_1pct', 0):.2f}  TP@1%: {set_metrics.get('tp_1pct', 0)}  Prec@1%: {set_metrics.get('precision_1pct', 0):.4f}")
        print(f"    Lift@5%: {set_metrics.get('lift_5pct', 0):.2f}  TP@5%: {set_metrics.get('tp_5pct', 0)}  Prec@5%: {set_metrics.get('precision_5pct', 0):.4f}")
        print(f"    Lift@10%: {set_metrics.get('lift_10pct', 0):.2f} TP@10%: {set_metrics.get('tp_10pct', 0)} Prec@10%: {set_metrics.get('precision_10pct', 0):.4f}")

## Data Loading

In [ ]:
# =============================================================================
# Load New TE Embeddings (256 dims)
# Source: Exp Round 10 / Exp2b (flash attention + learned pooling v2)
# =============================================================================

new_te_sql = f"""
    SELECT *
    FROM `{NEW_TE_TABLE}`
    WHERE index_dt BETWEEN '{TRAIN_START}' AND '{OOT_END}'
"""

print(f"Loading new TE embeddings from: {NEW_TE_TABLE}")
print(f"Date range: {TRAIN_START} to {OOT_END}")
df_new_te = client.query(new_te_sql).to_dataframe()
df_new_te['individual_id'] = df_new_te['individual_id'].astype(str)
print(f"New TE embeddings loaded: {df_new_te.shape}")
print(f"Columns: {list(df_new_te.columns[:10])}... ({len(df_new_te.columns)} total)")
print(f"index_dt range: {df_new_te['index_dt'].min()} to {df_new_te['index_dt'].max()}")

In [ ]:
# =============================================================================
# Load Production Features (52 baseline + 48 production TE embeddings)
# Source: Production inpatient_me_features_history table
# =============================================================================

# Build column list for production query
prod_emb_cols = ', '.join(PROD_EMB_COL_MAP.keys())  # emb6, emb7, ...
baseline_cols = ', '.join(BASELINE_FEATURES)

prod_sql = f"""
    SELECT
        individual_id,
        index_dt,
        {baseline_cols},
        {prod_emb_cols}
    FROM `{PRODUCTION_TABLE}`
    WHERE index_dt BETWEEN '{TRAIN_START}' AND '{OOT_END}'
"""

print(f"Loading production features from: {PRODUCTION_TABLE}")
df_prod = client.query(prod_sql).to_dataframe()
df_prod['individual_id'] = df_prod['individual_id'].astype(str)
print(f"Production features loaded: {df_prod.shape}")
print(f"index_dt range: {df_prod['index_dt'].min()} to {df_prod['index_dt'].max()}")

# Rename production embedding columns to standardized names: emb6 -> embedding_6
df_prod = df_prod.rename(columns=PROD_EMB_COL_MAP)

# Add prefix to production embedding columns to distinguish from new TE
prod_emb_rename = {col: f'prod_{col}' for col in PRODUCTION_EMBEDDINGS}
df_prod = df_prod.rename(columns=prod_emb_rename)
PROD_EMB_FEATURES = [f'prod_{col}' for col in PRODUCTION_EMBEDDINGS]

print(f"Production embedding columns (renamed): {PROD_EMB_FEATURES[:5]}...")
print(f"Baseline feature columns: {BASELINE_FEATURES[:5]}...")

In [ ]:
# =============================================================================
# Load Outcomes (ip6, mon_6_include)
# Source: Generated by medicare_ip_outcome_generation_exp_round10_exp2b.sql
# =============================================================================

outcomes_sql = f"""
    SELECT
        individual_id,
        index_dt,
        ip6,
        sum_ip6_admits,
        sum_ip6_los,
        mon_6_include
    FROM `{OUTCOMES_TABLE}`
    WHERE mon_6_include = 1
"""

print(f"Loading outcomes from: {OUTCOMES_TABLE}")
df_outcomes = client.query(outcomes_sql).to_dataframe()
df_outcomes['individual_id'] = df_outcomes['individual_id'].astype(str)
print(f"Outcomes loaded: {df_outcomes.shape}")
print(f"IP rate: {df_outcomes['ip6'].mean():.4f} ({df_outcomes['ip6'].sum()} / {len(df_outcomes)})")
print(f"index_dt range: {df_outcomes['index_dt'].min()} to {df_outcomes['index_dt'].max()}")

In [ ]:
# =============================================================================
# Data Inspection
# =============================================================================

print("=" * 60)
print("DATA SOURCE SUMMARY")
print("=" * 60)

print(f"\nNew TE Embeddings:")
print(f"  Rows: {len(df_new_te):,}")
print(f"  Unique individuals: {df_new_te['individual_id'].nunique():,}")
print(f"  Embedding dims: {len([c for c in df_new_te.columns if c.startswith('embedding_')])}")

print(f"\nProduction Features:")
print(f"  Rows: {len(df_prod):,}")
print(f"  Unique individuals: {df_prod['individual_id'].nunique():,}")
print(f"  Baseline features: {len([c for c in df_prod.columns if c in BASELINE_FEATURES])}")
print(f"  Production embeddings: {len([c for c in df_prod.columns if c.startswith('prod_')])}")

print(f"\nOutcomes:")
print(f"  Rows: {len(df_outcomes):,}")
print(f"  Unique individuals: {df_outcomes['individual_id'].nunique():,}")
print(f"  IP rate: {df_outcomes['ip6'].mean():.4f}")

# Check overlap between data sources
new_te_ids = set(df_new_te['individual_id'].unique())
prod_ids = set(df_prod['individual_id'].unique())
outcome_ids = set(df_outcomes['individual_id'].unique())

overlap_all = new_te_ids & prod_ids & outcome_ids
print(f"\nIndividual overlap:")
print(f"  New TE only: {len(new_te_ids):,}")
print(f"  Production only: {len(prod_ids):,}")
print(f"  Outcomes only: {len(outcome_ids):,}")
print(f"  Overlap (all three): {len(overlap_all):,}")

## Preprocessing & Merging

In [ ]:
# =============================================================================
# Merge all data sources on (individual_id, index_dt)
# Inner join ensures same population across all 3 models for fair comparison
# =============================================================================

# Step 1: Merge new TE embeddings with outcomes
df_merged = df_new_te.merge(
    df_outcomes[['individual_id', 'index_dt', 'ip6', 'sum_ip6_admits', 'sum_ip6_los']],
    on=['individual_id', 'index_dt'],
    how='inner'
)
print(f"After merging new TE + outcomes: {df_merged.shape}")

# Step 2: Merge with production features (baseline + production embeddings)
df_merged = df_merged.merge(
    df_prod,
    on=['individual_id', 'index_dt'],
    how='inner'
)
print(f"After merging with production features: {df_merged.shape}")

# Free memory
del df_new_te, df_prod, df_outcomes
gc.collect()

# Preprocess
df_merged = preprocess_dataframe(df_merged)

print(f"\nFinal merged dataset: {df_merged.shape}")
print(f"IP rate: {df_merged['ip6'].mean():.4f} ({df_merged['ip6'].sum()} / {len(df_merged)})")
print(f"index_dt range: {df_merged['index_dt'].min()} to {df_merged['index_dt'].max()}")

In [ ]:
# Verify all required feature columns exist
print("Feature availability check:")

# New TE embeddings (all 256)
missing_new_te = [c for c in ALL_NEW_TE_EMBEDDINGS if c not in df_merged.columns]
print(f"  New TE (256 dims): {len(ALL_NEW_TE_EMBEDDINGS) - len(missing_new_te)}/256 available")
if missing_new_te:
    print(f"    Missing: {missing_new_te[:10]}...")

# New TE selected (48)
missing_selected = [c for c in PRODUCTION_EMBEDDINGS if c not in df_merged.columns]
print(f"  New TE selected (48 dims): {len(PRODUCTION_EMBEDDINGS) - len(missing_selected)}/48 available")

# Baseline features (52)
missing_baseline = [c for c in BASELINE_FEATURES if c not in df_merged.columns]
print(f"  Baseline features (52): {len(BASELINE_FEATURES) - len(missing_baseline)}/52 available")
if missing_baseline:
    print(f"    Missing: {missing_baseline}")

# Production embeddings (48)
missing_prod_emb = [c for c in PROD_EMB_FEATURES if c not in df_merged.columns]
print(f"  Production TE (48 dims): {len(PROD_EMB_FEATURES) - len(missing_prod_emb)}/48 available")
if missing_prod_emb:
    print(f"    Missing: {missing_prod_emb[:10]}...")

## Feature Set Definition

In [ ]:
# =============================================================================
# Define the 3 feature sets for comparison
# =============================================================================

FEATURE_SETS = {
    'Model 1: New TE Only (256 dims)': {
        'features': ALL_NEW_TE_EMBEDDINGS,
        'description': 'Exp Round 10 / Exp2b embeddings only (flash attention + learned pooling v2)',
        'config_type': 'embeddings_only',
        'scale': True,
    },
    'Model 2: New TE Selected + Baseline': {
        'features': PRODUCTION_EMBEDDINGS + BASELINE_FEATURES,
        'description': 'New TE (48 production-selected dims) + 52 production baseline features',
        'config_type': 'hybrid',
        'scale': False,
    },
    'Model 3: Production TE + Baseline': {
        'features': PROD_EMB_FEATURES + BASELINE_FEATURES,
        'description': 'Production TE (48 dims) + 52 production baseline features',
        'config_type': 'production',
        'scale': False,
    },
}

for name, config in FEATURE_SETS.items():
    print(f"\n{name}:")
    print(f"  Features: {len(config['features'])}")
    print(f"  Config: {config['config_type']}")
    print(f"  Scale: {config['scale']}")
    print(f"  Description: {config['description']}")

## Temporal Split & Train/Val/Test Split

In [ ]:
# =============================================================================
# Temporal Split: In-Time vs Out-of-Time
# =============================================================================

# Convert index_dt to datetime for comparison
df_merged['index_dt'] = pd.to_datetime(df_merged['index_dt'])

# In-time: training period (2024-07-01 to 2025-06-30)
mask_intime = (df_merged['index_dt'] >= TRAIN_START) & (df_merged['index_dt'] <= TRAIN_END)
df_intime = df_merged[mask_intime].copy()

# OOT: out-of-time test period (2025-07-01 to 2025-09-30)
mask_oot = (df_merged['index_dt'] >= OOT_START) & (df_merged['index_dt'] <= OOT_END)
df_oot = df_merged[mask_oot].copy()

print(f"Temporal Split:")
print(f"  In-time ({TRAIN_START} to {TRAIN_END}): {len(df_intime):,} rows, IP rate: {df_intime['ip6'].mean():.4f}")
print(f"  OOT ({OOT_START} to {OOT_END}): {len(df_oot):,} rows, IP rate: {df_oot['ip6'].mean():.4f}")

# Free merged dataframe
del df_merged
gc.collect()

In [ ]:
# =============================================================================
# Stratified Random Split for In-Time Data: 70% Train / 15% Val / 15% Test
# Same as production pipeline (random_state=199, stratified on y)
# =============================================================================

y_intime = df_intime['ip6'].values
ids_intime = df_intime[['individual_id', 'index_dt']].copy()

# First split: 70% train, 30% temp (val + test)
idx_train, idx_temp, y_train_split, y_temp = train_test_split(
    np.arange(len(df_intime)),
    y_intime,
    test_size=0.30,
    random_state=199,
    stratify=y_intime
)

# Second split: Split temp into 50% val, 50% test (15% each of total)
idx_val, idx_test, _, _ = train_test_split(
    idx_temp,
    y_temp,
    test_size=0.50,
    random_state=199,
    stratify=y_temp
)

# Store split indices for reuse across models
df_train = df_intime.iloc[idx_train].copy()
df_val = df_intime.iloc[idx_val].copy()
df_test = df_intime.iloc[idx_test].copy()

print(f"Stratified Random Split (random_state=199):")
print(f"  Train: {len(df_train):,} ({len(df_train)/len(df_intime)*100:.1f}%), IP rate: {df_train['ip6'].mean():.4f}")
print(f"  Val:   {len(df_val):,} ({len(df_val)/len(df_intime)*100:.1f}%), IP rate: {df_val['ip6'].mean():.4f}")
print(f"  Test:  {len(df_test):,} ({len(df_test)/len(df_intime)*100:.1f}%), IP rate: {df_test['ip6'].mean():.4f}")
print(f"  OOT:   {len(df_oot):,}, IP rate: {df_oot['ip6'].mean():.4f}")

# Free in-time full dataframe
del df_intime
gc.collect()

## Model Training

Train 3 XGBoost models, one for each feature set configuration.

- **Embeddings-only** (Model 1): max_depth=4, min_child_weight=50, eta=0.01, scale_pos_weight=14.4
- **Hybrid/Production** (Models 2 & 3): max_depth=3, min_child_weight=100, eta=0.005, heavy L1/L2

In [ ]:
def get_xgb_params(config_type):
    """Get XGBoost hyperparameters based on configuration type."""
    if config_type == 'embeddings_only':
        return {
            'objective': 'binary:logistic',
            'max_depth': 4,
            'min_child_weight': 50,
            'eta': 0.01,
            'subsample': 0.7,
            'colsample_bytree': 0.5,
            'lambda': 3.0,
            'alpha': 1.0,
            'gamma': 1.0,
            'scale_pos_weight': 14.4,
            'tree_method': 'hist',
            'grow_policy': 'depthwise',
            'eval_metric': 'auc',
            'seed': 123
        }
    else:  # hybrid or production
        return {
            'objective': 'binary:logistic',
            'max_depth': 3,
            'min_child_weight': 100,
            'eta': 0.005,
            'subsample': 0.6,
            'colsample_bytree': 0.4,
            'lambda': 5.0,
            'alpha': 2.0,
            'gamma': 2.0,
            'max_delta_step': 1,
            'tree_method': 'hist',
            'grow_policy': 'depthwise',
            'eval_metric': 'auc',
            'seed': 123
        }


def train_model(model_name, feature_config, df_train, df_val, df_test, df_oot):
    """Train a single XGBoost model and return metrics + model object."""
    features = feature_config['features']
    config_type = feature_config['config_type']
    scale = feature_config['scale']

    print(f"\n{'='*60}")
    print(f"Training: {model_name}")
    print(f"Features: {len(features)}, Config: {config_type}, Scale: {scale}")
    print(f"{'='*60}")

    # Extract features
    X_train = df_train[features].copy()
    X_val = df_val[features].copy()
    X_test = df_test[features].copy()
    X_oot = df_oot[features].copy()

    y_train = df_train['ip6'].values
    y_val = df_val['ip6'].values
    y_test = df_test['ip6'].values
    y_oot = df_oot['ip6'].values

    # StandardScaler for embeddings-only
    scaler = None
    if scale:
        scaler = StandardScaler()
        print(f"  Before scaling: mean={X_train.mean().mean():.3f}, std={X_train.std().mean():.3f}")
        X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=features, index=X_train.index)
        X_val = pd.DataFrame(scaler.transform(X_val), columns=features, index=X_val.index)
        X_test = pd.DataFrame(scaler.transform(X_test), columns=features, index=X_test.index)
        X_oot = pd.DataFrame(scaler.transform(X_oot), columns=features, index=X_oot.index)
        print(f"  After scaling:  mean={X_train.mean().mean():.3f}, std={X_train.std().mean():.3f}")

    # Create DMatrix
    dtrain = xgb.DMatrix(X_train, y_train)
    dval = xgb.DMatrix(X_val, y_val)
    dtest = xgb.DMatrix(X_test, y_test)
    doot = xgb.DMatrix(X_oot, y_oot)

    # Get hyperparameters
    params = get_xgb_params(config_type)
    print(f"\n  Params: max_depth={params['max_depth']}, eta={params['eta']}, "
          f"min_child_weight={params['min_child_weight']}")

    # Train
    start_time = time.time()
    bst = xgb.train(
        params,
        dtrain,
        num_boost_round=5000,
        evals=[(dtrain, 'train'), (dval, 'val')],
        early_stopping_rounds=100,
        verbose_eval=100
    )
    train_time = time.time() - start_time
    print(f"\n  Training time: {train_time:.1f}s, Best iteration: {bst.best_iteration}")

    # Predict
    pred_train = bst.predict(dtrain)
    pred_val = bst.predict(dval)
    pred_test = bst.predict(dtest)
    pred_oot = bst.predict(doot)

    # Calculate metrics
    metrics = {}
    metrics.update(calculate_comprehensive_metrics(y_train, pred_train, 'train'))
    metrics.update(calculate_comprehensive_metrics(y_val, pred_val, 'val'))
    metrics.update(calculate_comprehensive_metrics(y_test, pred_test, 'test'))
    metrics.update(calculate_comprehensive_metrics(y_oot, pred_oot, 'oot'))
    metrics['best_iteration'] = bst.best_iteration
    metrics['train_time_sec'] = train_time

    print_metrics_summary(metrics, model_name)

    return {
        'model': bst,
        'metrics': metrics,
        'scaler': scaler,
        'params': params,
        'features': features,
        'predictions': {
            'train': pred_train,
            'val': pred_val,
            'test': pred_test,
            'oot': pred_oot,
        },
        'y_true': {
            'train': y_train,
            'val': y_val,
            'test': y_test,
            'oot': y_oot,
        }
    }

In [ ]:
# =============================================================================
# Train all 3 models
# =============================================================================

results = {}

for model_name, feature_config in FEATURE_SETS.items():
    results[model_name] = train_model(
        model_name, feature_config, df_train, df_val, df_test, df_oot
    )

print(f"\n\nAll 3 models trained successfully.")

## In-Time Comparison (Test Set)

In [ ]:
# =============================================================================
# Summary Comparison Table — In-Time Test Set
# =============================================================================

comparison_metrics = ['auc_roc', 'auc_pr', 'brier', 'lift_1pct', 'lift_5pct', 'lift_10pct',
                      'tp_1pct', 'tp_5pct', 'tp_10pct', 'precision_1pct', 'precision_5pct', 'precision_10pct']

rows = []
for model_name, res in results.items():
    row = {'Model': model_name}
    row['Features'] = len(res['features'])
    row['Best Iter'] = res['metrics']['best_iteration']
    for metric in comparison_metrics:
        row[f'Test {metric}'] = res['metrics'].get(f'test_{metric}', None)
        row[f'Val {metric}'] = res['metrics'].get(f'val_{metric}', None)
    rows.append(row)

df_comparison = pd.DataFrame(rows)

print("\n" + "=" * 80)
print("IN-TIME COMPARISON (Test Set)")
print("=" * 80)

# Display key metrics
display_cols = ['Model', 'Features', 'Test auc_roc', 'Test auc_pr', 'Test brier',
                'Test lift_1pct', 'Test lift_5pct', 'Test tp_1pct', 'Test tp_5pct']
print(df_comparison[display_cols].to_string(index=False))

In [ ]:
# =============================================================================
# Visualization — In-Time: ROC, PR, and Lift Curves
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
colors = ['#2196F3', '#FF9800', '#4CAF50']

for i, (model_name, res) in enumerate(results.items()):
    y_true = res['y_true']['test']
    y_pred = res['predictions']['test']
    short_name = model_name.split(':')[0].strip()

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    auc_val = roc_auc_score(y_true, y_pred)
    axes[0].plot(fpr, tpr, color=colors[i], label=f'{short_name} (AUC={auc_val:.4f})', linewidth=2)

    # PR Curve
    prec, rec, _ = precision_recall_curve(y_true, y_pred)
    ap_val = average_precision_score(y_true, y_pred)
    axes[1].plot(rec, prec, color=colors[i], label=f'{short_name} (AP={ap_val:.4f})', linewidth=2)

    # Lift Curve (at various percentiles)
    pcts = np.arange(0.01, 0.21, 0.01)
    lifts = [lift_at_percentage(y_true, y_pred, p) for p in pcts]
    axes[2].plot(pcts * 100, lifts, color=colors[i], label=f'{short_name}', linewidth=2, marker='o', markersize=3)

# Formatting
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve — In-Time Test')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve — In-Time Test')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

axes[2].axhline(y=1.0, color='k', linestyle='--', alpha=0.3, label='Baseline')
axes[2].set_xlabel('Top K% of Population')
axes[2].set_ylabel('Lift')
axes[2].set_title('Lift Curve — In-Time Test')
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.suptitle('In-Time Test Set: 3-Model Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Out-of-Time (OOT) Comparison

In [ ]:
# =============================================================================
# Summary Comparison Table — OOT
# =============================================================================

rows_oot = []
for model_name, res in results.items():
    row = {'Model': model_name}
    row['Features'] = len(res['features'])
    for metric in comparison_metrics:
        row[f'OOT {metric}'] = res['metrics'].get(f'oot_{metric}', None)
    rows_oot.append(row)

df_oot_comparison = pd.DataFrame(rows_oot)

print("\n" + "=" * 80)
print(f"OUT-OF-TIME COMPARISON ({OOT_START} to {OOT_END})")
print("=" * 80)

display_cols_oot = ['Model', 'Features', 'OOT auc_roc', 'OOT auc_pr', 'OOT brier',
                    'OOT lift_1pct', 'OOT lift_5pct', 'OOT tp_1pct', 'OOT tp_5pct']
print(df_oot_comparison[display_cols_oot].to_string(index=False))

In [ ]:
# =============================================================================
# Visualization — OOT: ROC, PR, and Lift Curves
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, (model_name, res) in enumerate(results.items()):
    y_true = res['y_true']['oot']
    y_pred = res['predictions']['oot']
    short_name = model_name.split(':')[0].strip()

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_true, y_pred)
    auc_val = roc_auc_score(y_true, y_pred)
    axes[0].plot(fpr, tpr, color=colors[i], label=f'{short_name} (AUC={auc_val:.4f})', linewidth=2)

    # PR Curve
    prec, rec, _ = precision_recall_curve(y_true, y_pred)
    ap_val = average_precision_score(y_true, y_pred)
    axes[1].plot(rec, prec, color=colors[i], label=f'{short_name} (AP={ap_val:.4f})', linewidth=2)

    # Lift Curve
    pcts = np.arange(0.01, 0.21, 0.01)
    lifts = [lift_at_percentage(y_true, y_pred, p) for p in pcts]
    axes[2].plot(pcts * 100, lifts, color=colors[i], label=f'{short_name}', linewidth=2, marker='o', markersize=3)

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve — OOT')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve — OOT')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

axes[2].axhline(y=1.0, color='k', linestyle='--', alpha=0.3, label='Baseline')
axes[2].set_xlabel('Top K% of Population')
axes[2].set_ylabel('Lift')
axes[2].set_title('Lift Curve — OOT')
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'OOT Test ({OOT_START} to {OOT_END}): 3-Model Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# Combined: In-Time vs OOT Side-by-Side
# =============================================================================

print("\n" + "=" * 100)
print("COMBINED COMPARISON: In-Time Test vs OOT")
print("=" * 100)

combined_rows = []
for model_name, res in results.items():
    short_name = model_name.split(':')[1].strip() if ':' in model_name else model_name
    row = {
        'Model': short_name,
        'N Features': len(res['features']),
        'Test AUC-ROC': f"{res['metrics']['test_auc_roc']:.4f}",
        'OOT AUC-ROC': f"{res['metrics']['oot_auc_roc']:.4f}",
        'Delta AUC': f"{res['metrics']['oot_auc_roc'] - res['metrics']['test_auc_roc']:+.4f}",
        'Test AUC-PR': f"{res['metrics']['test_auc_pr']:.4f}",
        'OOT AUC-PR': f"{res['metrics']['oot_auc_pr']:.4f}",
        'Test Lift@5%': f"{res['metrics']['test_lift_5pct']:.2f}",
        'OOT Lift@5%': f"{res['metrics']['oot_lift_5pct']:.2f}",
        'Test TP@5%': res['metrics']['test_tp_5pct'],
        'OOT TP@5%': res['metrics']['oot_tp_5pct'],
    }
    combined_rows.append(row)

df_combined = pd.DataFrame(combined_rows)
print(df_combined.to_string(index=False))

## Feature Importance Analysis

In [ ]:
# =============================================================================
# Feature Importance: Top 20 features per model
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(24, 8))

for i, (model_name, res) in enumerate(results.items()):
    importance = res['model'].get_score(importance_type='gain')
    if not importance:
        axes[i].text(0.5, 0.5, 'No importance data', ha='center', va='center')
        continue

    # Sort and get top 20
    sorted_imp = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:20]
    features_top = [x[0] for x in sorted_imp][::-1]
    values_top = [x[1] for x in sorted_imp][::-1]

    axes[i].barh(range(len(features_top)), values_top, color=colors[i], alpha=0.8)
    axes[i].set_yticks(range(len(features_top)))
    axes[i].set_yticklabels(features_top, fontsize=8)
    short_name = model_name.split(':')[0].strip()
    axes[i].set_title(f'{short_name} — Top 20 (gain)', fontsize=11)
    axes[i].set_xlabel('Gain')
    axes[i].grid(True, alpha=0.3, axis='x')

plt.suptitle('Feature Importance by Gain', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Save Artifacts

In [ ]:
# =============================================================================
# Save models, metrics, and comparison tables
# =============================================================================

output_dir = 'artifacts/medicare_ip_full_downstream_eval'
os.makedirs(output_dir, exist_ok=True)

# Save each model
model_names_short = {
    'Model 1: New TE Only (256 dims)': 'model1_new_te_only',
    'Model 2: New TE Selected + Baseline': 'model2_new_te_baseline',
    'Model 3: Production TE + Baseline': 'model3_production',
}

for model_name, res in results.items():
    short = model_names_short[model_name]
    # Save XGBoost model
    res['model'].save_model(f'{output_dir}/{short}.json')
    # Save scaler if exists
    if res['scaler'] is not None:
        joblib.dump(res['scaler'], f'{output_dir}/{short}_scaler.pkl')
    print(f"Saved: {short}")

# Save comparison tables
df_comparison.to_csv(f'{output_dir}/comparison_intime.csv', index=False)
df_oot_comparison.to_csv(f'{output_dir}/comparison_oot.csv', index=False)
df_combined.to_csv(f'{output_dir}/comparison_combined.csv', index=False)

# Save all metrics as pickle
all_metrics = {name: res['metrics'] for name, res in results.items()}
with open(f'{output_dir}/all_metrics.pkl', 'wb') as f:
    pickle.dump(all_metrics, f)

print(f"\nAll artifacts saved to: {output_dir}/")

## Summary

### Three-Model Comparison: Medicare IP 6-Month Prediction

| Component | Model 1 | Model 2 | Model 3 |
|-----------|---------|---------|--------|
| **Embeddings** | New TE (256 dims) | New TE (48 selected) | Production TE (48 dims) |
| **Baseline** | None | 52 production features | 52 production features |
| **Total Features** | 256 | 100 | 100 |
| **XGB Config** | Embeddings-only | Hybrid | Production |
| **StandardScaler** | Yes | No | No |

**Key Differences**:
- **Model 1** tests pure embedding representation power (new TE, all 256 dims)
- **Model 2** tests new TE compatibility with production baseline (48 selected dims + 52 features)
- **Model 3** is the production benchmark (production TE + production baseline)

**Temporal Design**:
- Training: 2024-07-01 to 2025-06-30 (70/15/15 stratified split)
- OOT: 2025-07-01 to 2025-09-30 (temporal generalization test)

**Same population** across all 3 models (inner join on individual_id + index_dt).